# Curriculum-Based AI Tutor using RAG

## Project Overview

This project implements a Retrieval-Augmented Generation (RAG) based AI Tutor for students using the NCERT Class 8 Science textbook. The system combines semantic search with a Large Language Model (LLM) to answer textbook-related questions in simple and grade-appropriate language.

The project workflow includes:

- Extracting and cleaning textbook text from PDF
- Creating embeddings using Sentence Transformers
- Storing embeddings in a FAISS vector database
- Building a retrieval-based question answering pipeline
- Generating answers using an LLM
- Providing source citations for transparency
- Evaluating responses using BLEU and ROUGE-L metrics

## Objectives

- Build an educational AI tutor focused on NCERT Class 8 Science
- Reduce hallucination using retrieval-augmented generation
- Restrict answers to textbook content only
- Evaluate answer quality using automatic metrics and human review

## Technologies Used

- Python
- PyMuPDF
- Sentence Transformers
- FAISS
- LangChain
- Llama-2
- Streamlit
- NLTK
- BLEU, ROUGE Score

## Dataset

Source textbook:
NCERT Class 8 Science Textbook

The corpus is limited to 13 chapters from the official NCERT textbook.

## Expected Deliverables

- Cleaned textbook corpus
- Vector database and retriever
- RAG pipeline notebook
- Evaluation report
- Streamlit/Gradio demo application

---

### Step-1 Install Required Libraries

In [ ]:
import sys

# Upgrade pip first
!{sys.executable} -m pip install --upgrade pip

# Install compatible versions for project
!{sys.executable} -m pip install transformers==4.41.2
!{sys.executable} -m pip install peft==0.10.0
!{sys.executable} -m pip install tokenizers==0.19.1
!{sys.executable} -m pip install accelerate==0.30.1
!{sys.executable} -m pip install sentence-transformers
!{sys.executable} -m pip install datasets
!{sys.executable} -m pip install faiss-cpu
!{sys.executable} -m pip install langchain
!{sys.executable} -m pip install haystack-ai
!{sys.executable} -m pip install evaluate
!{sys.executable} -m pip install rouge-score
!{sys.executable} -m pip install nltk
!{sys.executable} -m pip install fsspec==2026.2.0

### Step-2 Extract PDF Text

In [1]:
import fitz  # PyMuPDF

pdf_path = r"C:\Users\hp\Downloads\BIA\curriculum_ai_tutor\data\raw\class8_science.pdf"

doc = fitz.open(pdf_path)

full_text = ""

for page in doc:
    text = page.get_text()
    full_text += text + "\n"

print(full_text[:2000])

Curiosity
Curiosity
Textbook of Science for Grade 8
Reprint 2026-27

First Edition
June 2025  Ashadha 1947
Reprinted
January 2026  Magha 1947
PD 730T SM
© National Council of Educational 
Research and Training, 2025
` 65.00
ALL RIGHTS RESERVED
	 No part of this publication may be reproduced, stored in a retrieval system 
or transmitted, in any form or by any means, electronic, mechanical, 
photocopying, recording or otherwise without the prior permission of the 
publisher.
	 This book is sold subject to the condition that it shall not, by way of trade,  be lent, 
 re-sold, hired out or otherwise disposed of without the publisher’s consent, in 
any form of binding or cover other than that in which it is published.
	 The correct price of this publication is the price printed on this page, Any 
revised price indicated by a rubber stamp or by a sticker or by any other 
means is incorrect and should be unacceptable.
OFFICES OF THE PUBLICATION 
DIVISION, NCERT
NCERT Campus
Sri Aurobindo M

### Step-3 Clean and Split Text
- Goal:
- Remove headers/footers/page numbers
- Split by Chapter
- Create Chunks


In [2]:
# Cleaning Function
import re

def clean_text(text):

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    # Remove symbols
    text = re.sub(r'[•●◆■]', '', text)

    # Remove ISBN
    text = re.sub(r'ISBN\s[\d\-]+', '', text)

    # Start only from Chapter 1
    start = text.find("Chapter 1 ")

    if start != -1:
        text = text[start:]

    return text.strip()


cleaned_text = clean_text(full_text)

real_start = cleaned_text.find("Chapter 1 — Exploring")

cleaned_text = cleaned_text[real_start:]

print(cleaned_text[:3000])

# Remove Constitution section
constitution_start = cleaned_text.find("CONSTITUTION OF INDIA")

chapter_start = cleaned_text.find("Chapter 1 — Exploring")

if constitution_start != -1 and chapter_start != -1:
    cleaned_text = (
        cleaned_text[:constitution_start] +
        cleaned_text[chapter_start:]
    )

print(cleaned_text[:3000])

Chapter 1 — Exploring the Investigative World of Science 1 Exploring the Investigative World of Science 1 Dear Young Scientists, Welcome back! On the first page of each chapter, you will find a set of questions. These are not meant for any exam— they are unique invitations to spark your curiosity to explore the world of science! Why is one side of a puri thinner than the other? Are there more grains of sand on all the beaches and deserts of the world, or more stars in our galaxy? Right from Grade 6, we’ve observed the incredible diversity of plants and animals around us. From the different shapes of leaves to the many kinds of insects — why has nature created such a vast variety? Is there such a question that makes you curious about the world? Write it here! _________________________________ Probe and ponder Reprint 2026-27 2 Curiosity — Textbook of Science for Grade 8 Our journey with Curiosity, into the world of science continues in Grade 8. We hope you bring along the spirit of adve

In [3]:
# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_text(cleaned_text)

print("Total chunks:", len(chunks))
print(chunks[0])

Total chunks: 939
Chapter 1 — Exploring the Investigative World of Science 1 Exploring the Investigative World of Science 1 Dear Young Scientists, Welcome back! On the first page of each chapter, you will find a set of questions. These are not meant for any exam— they are unique invitations to spark your curiosity to explore the world of science! Why is one side of a puri thinner than the other? Are there more grains of sand on all the beaches and deserts of the world, or more stars in our galaxy? Right from


### Save JSONL Corpus

In [42]:
import os
import json

with open("../data/processed/class8_science.jsonl", "w", encoding="utf-8") as f:

    for idx, chunk in enumerate(chunks):

        record = {
            "chunk_id": idx,
            "text": chunk
        }

        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("JSONL file saved successfully.")

JSONL file saved successfully.


### Step-4 Create Embeddings and FAISS Index

In [5]:
# Generate Embeddings
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

embeddings = embedding_model.encode(chunks)

print(embeddings.shape)

C:\Users\hp\aitutor_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


(939, 384)


In [6]:
# Create FAISS Index
import faiss
import numpy as np
import os

# Create models folder if not exists
os.makedirs("../models", exist_ok=True)

# Embedding dimension
dimension = embeddings.shape[1]

# Create FAISS index
index = faiss.IndexFlatL2(dimension)

# Add embeddings
index.add(np.array(embeddings))

# Save index
faiss.write_index(
    index,
    "../models/science_index.faiss"
)

print("FAISS index saved successfully.")

FAISS index saved successfully.


### Step-5 Build RAG Pipeline

In [7]:
# Create LangChain Vector Store
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = FAISS.from_texts(
    chunks,
    embedding_function
)

C:\Users\hp\AppData\Local\Temp\ipykernel_13764\3909641103.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
C:\Users\hp\AppData\Local\Temp\ipykernel_13764\3909641103.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = HuggingFaceEmbeddings(
C:\Users\hp\aitutor_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads a

In [8]:
# Save FAISS Vector Store
vectorstore.save_local("class8_vectorstore")

print("Vector store saved successfully!")

Vector store saved successfully!


In [9]:
# Create Retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 5
    }
)

In [10]:
# Test LangChain Retriever
docs = retriever.invoke(
    "What is photosynthesis?"
)

for doc in docs:
    print(doc.page_content)
    print("\n-------------------\n")

and animals. Now, let us explore how all these elements interact to support and sustain life on Earth. 13.4.1 Air, water, and sunlight We know that the atmosphere contains oxygen, which humans, animals, and plants use for respiration. In the presence of sunlight, plants take carbon dioxide from the air and water from Reprint 2026-27 218 Curiosity — Textbook of Science for Grade 8 the soil to prepare food by photosynthesis. In the process, oxygen is released, which is needed for respiration. We

-------------------

within the cell. It also regulates growth. The cell wall in the plant cell provides rigidity and strength to plants. This is why all cells are arranged compactly with each other and look firm in structure. A step further Cells in all parts of a plant have tiny rod-shaped structures called plastids. Some plastids, like chloroplasts, contain chlorophyll, which makes them green and helps in photosynthesis. In non-green parts, they help in the storage of substances. Plant cells 

In [11]:
# Load LLAMA-2-7B-Q
from langchain_community.llms import CTransformers

llm = CTransformers(
    model="../models/llama-2-7b-chat.Q4_K_M.gguf",
    model_type="llama",
    config={
        "max_new_tokens": 120,
        "temperature": 0.0,
        "top_p": 1.0,
        "repetition_penalty": 1.15,
        "context_length": 4096
    }
)

In [12]:
# Create Prompt Template
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an NCERT Class 8 Science tutor.

Rules:
- Answer ONLY using the provided context.
- Do not add outside knowledge.
- Keep answers short and textbook-like.
- Keep the answer within 2 sentences.
- Use simple language suitable for Class 8 students.
- Use only Class 8 NCERT-level terminology.
- Avoid advanced biological terms unless explicitly present in context.
- Include all important points.
- For hierarchies, include every level.
- For comparisons, include all differences.
- If the answer is not found in the context, say:
  "I’m focused on Class 8 Science NCERT content."
- Do not explain beyond the textbook.

Context:
{context}

Question:
{input}

Answer:
""")

In [13]:
# Create RAG Chain
from langchain_core.runnables import RunnablePassthrough

from langchain_core.output_parsers import StrOutputParser

In [14]:
# Build Pipeline Manually
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [15]:
# Run Query
result = rag_chain.invoke("What is photosynthesis?")
print(result)

Photosynthesis is the process in which plants take carbon dioxide from the air and water from the soil to prepare food by releasing oxygen and storing energy.


### Step-6 Add Source Citations

In [16]:
docs = retriever.invoke("What is photosynthesis?")

for i, d in enumerate(docs):
    print(f"\nSOURCE {i+1}")
    print(d.page_content)


SOURCE 1
and animals. Now, let us explore how all these elements interact to support and sustain life on Earth. 13.4.1 Air, water, and sunlight We know that the atmosphere contains oxygen, which humans, animals, and plants use for respiration. In the presence of sunlight, plants take carbon dioxide from the air and water from Reprint 2026-27 218 Curiosity — Textbook of Science for Grade 8 the soil to prepare food by photosynthesis. In the process, oxygen is released, which is needed for respiration. We

SOURCE 2
within the cell. It also regulates growth. The cell wall in the plant cell provides rigidity and strength to plants. This is why all cells are arranged compactly with each other and look firm in structure. A step further Cells in all parts of a plant have tiny rod-shaped structures called plastids. Some plastids, like chloroplasts, contain chlorophyll, which makes them green and helps in photosynthesis. In non-green parts, they help in the storage of substances. Plant cells al

### Step- 7 Interaction Logging

In [17]:
#Log all interactions for later error analysis
import pandas as pd
from datetime import datetime
import os

LOG_FILE = "self_interaction_logs.csv"

def log_interaction(question, answer):

    log_data = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "question": question,
        "answer": answer
    }

    df = pd.DataFrame([log_data])

    if os.path.exists(LOG_FILE):
        df.to_csv(LOG_FILE, mode='a', header=False, index=False)
    else:
        df.to_csv(LOG_FILE, index=False)

### Step-7 Build Final QA Function

In [18]:
def ask_question(query):

    # Retrieve docs
    docs = retriever.invoke(query)

    # Combine context
    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    # Generate answer
    formatted_prompt = prompt.format(
        context=context,
        input=query
    )

    answer = llm.invoke(formatted_prompt)

    # Log interaction
    log_interaction(query, answer)

    print("\nQUESTION:\n")
    print(query)

    print("\nANSWER:\n")
    print(answer)

    print("\nSOURCES:\n")

    for i, doc in enumerate(docs):

        print(f"\nSource {i+1}:")
        print(doc.page_content[:300])

        print("\n-------------------")

In [20]:
# Test It
ask_question(" Difference between manure and fertilizer.")


QUESTION:

 Difference between manure and fertilizer.

ANSWER:

Manure is organic matter that has been broken down by microorganisms into a dark-colored material rich in nutrients, while fertilizers are synthetic or natural substances added to soil to promote plant growth. Manure provides slow release of nutrients to plants over time, whereas fertilizers provide quick release of nutrients.

SOURCES:


Source 1:
leaves into simpler substances that are rich in nutrients — a process called decomposition. These nutrients go back to the soil and help plants grow better. Microorganisms also decompose bodies of dead animals. So, microbes help recycle the waste and return important nutrients to nature. Manure form

-------------------

Source 2:
added nitrogen-rich fertiliser to the soil to get a good yield of crops. In the neighbouring field, another farmer was growing bean crops, but she preferred not to add nitrogen fertiliser to get healthy crops. Can you think of the reasons? 4. Snehal d

### Step-8 Add Out-of-Syllabus Detection

In [27]:
import re

def ask_question(query):

    # Retrieve docs with scores
    results = vectorstore.similarity_search_with_score(
        query,
        k=3
    )

    # No results found
    if len(results) == 0:

        answer = (
            "I'm focused on Class 8 Science NCERT content; try re-phrasing."
        )

        log_interaction(query, answer)

        print("\nQUESTION:\n")
        print(query)

        print("\nANSWER:\n")
        print(answer)

        return

    # Best similarity score
    best_score = results[0][1]

    print("\nDEBUG INFO")
    print("Best Score:", best_score)

    # ----------------------------------
    # Keyword overlap check
    # ----------------------------------

    stop_words = {
        "what", "who", "when", "where", "why",
        "how", "is", "are", "was", "were",
        "the", "a", "an", "of", "in", "on",
        "for", "to", "and", "or"
    }

    query_words = {
        re.sub(r'[^a-zA-Z0-9]', '', word.lower())
        for word in query.split()
    }

    query_words = {
        word
        for word in query_words
        if len(word) > 2 and word not in stop_words
    }

    keyword_found = False

    for doc, score in results:

        content_words = set(
            re.findall(
                r'\b\w+\b',
                doc.page_content.lower()
            )
        )

        if query_words.intersection(content_words):
            keyword_found = True
            break

    print("Keyword Found:", keyword_found)

    # ----------------------------------
    # Out-of-syllabus detection
    # ----------------------------------

    if best_score > 1.3 or not keyword_found:

        answer = (
            "I'm focused on Class 8 Science NCERT content; try re-phrasing."
        )

        print("\nQUESTION:\n")
        print(query)

        print("\nANSWER:\n")
        print(answer)

        log_interaction(
            query,
            answer
        )

        return

    # Extract documents
    docs = [
        doc
        for doc, score in results
    ]

    # Combine context
    context = "\n\n".join(
        [
            doc.page_content
            for doc in docs
        ]
    )

    # Format prompt
    formatted_prompt = prompt.format(
        context=context,
        input=query
    )

    # Generate answer
    answer = llm.invoke(
        formatted_prompt
    )

    # Log interaction
    log_interaction(
        query,
        answer
    )

    print("\nQUESTION:\n")
    print(query)

    print("\nANSWER:\n")
    print(answer)

    print("\nSOURCES:\n")

    for i, (doc, score) in enumerate(results):

        print(f"\nSource {i+1}")
        print("Score:", score)

        print(doc.page_content[:300])

        print("\n-------------------")

In [30]:
ask_question("How are antibiotics useful ?")


DEBUG INFO
Best Score: 0.82600546
Keyword Found: True

QUESTION:

How are antibiotics useful ?

ANSWER:

Antibiotics are useful because they kill or stop the growth of harmful bacteria that can cause diseases. They work by targeting specific parts of bacterial cells, which are different from human or other animal cells. This makes them effective against bacterial infections but not viruses or diseases caused by protozoa.

SOURCES:


Source 1
Score: 0.82600546
and keeps antibiotics effective for future generations. Traditional medicine systems like Ayurveda, Siddha, and Unani have been used in India for many years to manage common health problems. They use natural substances like herbs, oils, HOW BACTERIA BECAME RESISTANT TO ANTIBIOTICS? I need antibiotic

-------------------

Source 2
Score: 0.8425328
may give us medicines called antibiotics, which kill the bacteria that might have caused the disease. Antibiotics work only against bacterial infections because they target parts of bact

### Step-9 Create Evaluation Set

In [35]:
# Create Test Questions 
test_data = [
    {
 "question":"What is a cell?",
 "expected_answer":"A cell is the basic unit of life."
},

{
 "question":"What are microorganisms?",
 "expected_answer":"Microorganisms are tiny living organisms that cannot be seen with the naked eye."
},

{
 "question":"What is cytoplasm?",
 "expected_answer":"Cytoplasm is the jelly-like substance present between the cell membrane and nucleus where most life processes occur."
},

{
 "question":"What is a solution?",
 "expected_answer":"A solution is a homogeneous mixture formed when a solute dissolves in a solvent."
},

{
 "question":"What is force?",
 "expected_answer":"Force is a push or pull that can change the state of motion of an object."
},
{
 "question":"Why do plant cells have a cell wall?",
 "expected_answer":"The cell wall provides rigidity, strength and support to plant cells."
},

{
 "question":"What are the levels of organisation in living organisms?",
 "expected_answer":"Cell, tissue, organ, organ system and organism are the levels of organisation."
},

{
 "question":"Why are nerve cells branched?",
 "expected_answer":"Their branched structure helps them carry messages to different parts of the body."
},

{
 "question":"How does a microscope help scientists?",
 "expected_answer":"A microscope magnifies tiny objects and allows scientists to observe structures that cannot be seen with the naked eye."
},

{
 "question":"What is the function of the nucleus?",
 "expected_answer":"The nucleus regulates the activities and growth of the cell."
},
{
 "question":"Compare plant cells and animal cells.",
 "expected_answer":"Plant cells have a cell wall, chloroplasts and a large vacuole while animal cells do not."
},

{
 "question":"How are cells organised to form a living organism?",
 "expected_answer":"Cells form tissues, tissues form organs, organs form organ systems and organ systems form an organism."
},

{
 "question":"Why are muscle cells spindle-shaped?",
 "expected_answer":"Their shape helps them contract and relax to perform movement."
},

{
 "question":"Why are microorganisms important?",
 "expected_answer":"Some microorganisms are useful in nature and human life, while others may cause diseases."
},

{
 "question":"Why is scientific investigation important?",
 "expected_answer":"Scientific investigation helps us observe, experiment and understand the world systematically."
},
{
 "question":"What is machine learning?",
 "expected_answer":"I'm focused on Class 8 Science NCERT content."
},

{
 "question":"Explain blockchain.",
 "expected_answer":"I'm focused on Class 8 Science NCERT content."
},

{
 "question":"What is ChatGPT?",
 "expected_answer":"I'm focused on Class 8 Science NCERT content."
},

{
 "question":"What is quantum computing?",
 "expected_answer":"I'm focused on Class 8 Science NCERT content."
},

{
 "question":"How do rockets use AI?",
 "expected_answer":"I'm focused on Class 8 Science NCERT content."
 }
]

In [22]:
# Install Required Libraries
!pip install rouge-score nltk

Defaulting to user installation because normal site-packages is not writeable
  Using cached rouge_score-0.1.2-py3-none-any.whl


In [36]:
# Import Required Libraries
import pandas as pd

from nltk.translate.bleu_score import sentence_bleu

from rouge_score import rouge_scorer    

In [37]:
# Create Evaluation Function
def ask_question_for_eval(query):

    # Retrieve documents with similarity scores
    results = vectorstore.similarity_search_with_score(
        query,
        k=3
    )

    # No results found
    if len(results) == 0:

        return (
            "I'm focused on Class 8 Science NCERT content."
        )

    # Best similarity score
    best_score = results[0][1]

    # Out-of-syllabus detection
    if best_score > 1.0:

        return (
            "I'm focused on Class 8 Science NCERT content."
        )

    # Extract documents
    docs = [doc for doc, score in results]

    # Combine context
    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    # Format prompt
    formatted_prompt = prompt.format(
        context=context,
        input=query
    )

    # Generate answer
    answer = llm.invoke(formatted_prompt)

    return answer.strip()

In [38]:
# Run Evaluation
evaluation_results = []

# Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(
    ['rougeL'],
    use_stemmer=True
)

for item in test_data:

    question = item["question"]
    expected = item["expected_answer"]

    # Generate answer
    print("\nGenerating answer...")

    generated = ask_question_for_eval(question)

    # Display outputs
    print("\n===================================")

    print("\nQUESTION:")
    print(question)

    print("\nEXPECTED ANSWER:")
    print(expected)

    print("\nGENERATED ANSWER:")
    print(generated)

    # BLEU Score
    bleu = sentence_bleu(
        [expected.split()],
        generated.split()
    )

    # ROUGE-L Score
    rouge = scorer.score(expected, generated)
    rouge_l = rouge['rougeL'].fmeasure

    print("\nBLEU:", round(bleu, 3))
    print("ROUGE-L:", round(rouge_l, 3))

    # Human Review
    review = input(
        f"\nEnter human review for:\n{question}\n"
    )

    print("\nReview saved!")
    print("\n===================================")

    # Store results
    evaluation_results.append({

        "Question": question,

        "Expected Answer": expected,

        "Generated Answer": generated,

        "BLEU": round(bleu, 3),

        "ROUGE-L": round(rouge_l, 3),

        "Reviewer Comment": review
    })


Generating answer...


QUESTION:
What is a cell?

EXPECTED ANSWER:
A cell is the basic unit of life.

GENERATED ANSWER:
A cell is the basic structural and functional unit of life. It is a complex structure made up of many different parts, each with its own special function to allow the cell, and in turn the entire organism to work.

BLEU: 0.115
ROUGE-L: 0.333



Enter human review for:
What is a cell?
 Accurate answer, includes additional relevant details.



Review saved!


Generating answer...


QUESTION:
What are microorganisms?

EXPECTED ANSWER:
Microorganisms are tiny living organisms that cannot be seen with the naked eye.

GENERATED ANSWER:
Microorganisms are living things that are too small to be seen with the naked eye. They include bacteria, fungi, and other tiny organisms that are found everywhere around us, including on plants, animals, soil, water, and even inside our own bodies.

BLEU: 0.143
ROUGE-L: 0.37



Enter human review for:
What are microorganisms?
 Correct answer with additional relevant details.



Review saved!


Generating answer...


QUESTION:
What is cytoplasm?

EXPECTED ANSWER:
Cytoplasm is the jelly-like substance present between the cell membrane and nucleus where most life processes occur.

GENERATED ANSWER:
Cytoplasm is the region between the nucleus and cell membrane where all life processes take place. It contains other components of the cell like carbohydrates, proteins, fats, and mineral salts.

BLEU: 0.0
ROUGE-L: 0.417


C:\Users\hp\aitutor_env\lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


... (34 outputs left out)


Enter human review for:
What is quantum computing?
 Correctly identified as out-of-syllabus and returned the expected fallback response.



Review saved!


Generating answer...


QUESTION:
How do rockets use AI?

EXPECTED ANSWER:
I'm focused on Class 8 Science NCERT content.

GENERATED ANSWER:
I'm focused on Class 8 Science NCERT content.

BLEU: 1.0
ROUGE-L: 1.0



Enter human review for:
How do rockets use AI?
 Correctly identified as out-of-syllabus and returned the expected fallback response.



Review saved!



In [39]:
# Create Evaluation DataFrame
evaluation_df = pd.DataFrame(
    evaluation_results
)

evaluation_df

,Question,Expected Answer,Generated Answer,BLEU,ROUGE-L,Reviewer Comment
0,What is a cell?,A cell is the basic unit of life.,A cell is the basic structural and functional ...,0.115,0.333,"Accurate answer, includes additional relevant ..."
1,What are microorganisms?,Microorganisms are tiny living organisms that ...,Microorganisms are living things that are too ...,0.143,0.370,Correct answer with additional relevant details.
2,What is cytoplasm?,Cytoplasm is the jelly-like substance present ...,Cytoplasm is the region between the nucleus an...,0.000,0.417,Correct answer with additional relevant details.
3,What is a solution?,A solution is a homogeneous mixture formed whe...,A solution is formed when a solid is mixed wit...,0.000,0.465,"Accurate answer, explains both solute and sol..."
4,What is force?,Force is a push or pull that can change the st...,Force is a push or pull that can change an obj...,0.175,0.375,Correct answer with additional relevant detail...
5,Why do plant cells have a cell wall?,"The cell wall provides rigidity, strength and ...",Plant cells have a cell wall to provide rigidi...,0.000,0.400,Mostly accurate response that explains the str...
6,What are the levels of organisation in living ...,"Cell, tissue, organ, organ system and organism...",The levels of organisations in living organism...,0.000,0.519,Accurately identifies all levels of biologica...
7,Why are nerve cells branched?,Their branched structure helps them carry mess...,Nerve cells or neurons have an elongated shape...,0.121,0.450,Correct answer with clear reasoning.
8,How does a microscope help scientists?,A microscope magnifies tiny objects and allows...,A microscope helps scientists observe subcellu...,0.000,0.281,Mostly accurate but overly specific.
9,What is the function of the nucleus?,The nucleus regulates the activities and growt...,The nucleus regulates all activities that occu...,0.000,0.343,Accurate and complete explanation.


In [40]:
# Save Evaluation Results to CSV
evaluation_df.to_csv(
    "evaluation.csv",
    index=False
)

print("evaluation.csv saved successfully!")

evaluation.csv saved successfully!


### Step-10 Calculate Overall Metrics

In [41]:
import pandas as pd

df = pd.read_csv("evaluation.csv")

avg_bleu = round(df["BLEU"].mean(), 3)
avg_rouge = round(df["ROUGE-L"].mean(), 3)

print("========== Evaluation Summary ==========")
print(f"Average BLEU Score    : {avg_bleu}")
print(f"Average ROUGE-L Score : {avg_rouge}")
print("========================================")

========== Evaluation Summary ==========
Average BLEU Score    : 0.287
Average ROUGE-L Score : 0.515


## Evaluation Results

The Curriculum-Based AI Tutor was evaluated using 20 test questions covering both in-syllabus and out-of-syllabus scenarios. Performance was measured using BLEU and ROUGE-L metrics, along with manual human review.

**Results:**

* Average BLEU Score: 0.287
* Average ROUGE-L Score: 0.515

The ROUGE-L score indicates that the generated answers showed good semantic overlap with the reference answers. The BLEU score was comparatively lower because the model frequently produced paraphrased or more detailed responses rather than exact textual matches.

Human evaluation showed that most answers were factually correct, aligned with NCERT Class 8 Science content, and written in language appropriate for middle-school students. The system also successfully handled out-of-syllabus queries by returning the predefined fallback response.
